In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/conagua_acuiferos_ags.csv",
    skiprows=6,
    dtype={"CLAVE": str},
    encoding="utf-8-sig",
)
df["CLAVE"] = df["CLAVE"].str.zfill(4)

print(df.shape)
df.info()
df

(5, 7)
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CLAVE      5 non-null      str    
 1   ACUÍFERO   5 non-null      str    
 2   R          5 non-null      float64
 3   DNC        5 non-null      float64
 4   VEAS       5 non-null      float64
 5   DMA        5 non-null      float64
 6   DOCUMENTO  5 non-null      str    
dtypes: float64(4), str(3)
memory usage: 412.0 bytes


,CLAVE,ACUÍFERO,R,DNC,VEAS,DMA,DOCUMENTO
0,0101,VALLE DE AGUASCALIENTES,249.6,2.4,342.958309,-95.758309,DR_0101
1,0102,VALLE DE CHICALOTE,35.0,0.0,48.796081,-13.796081,DR_0102
2,0103,EL LLANO,15.0,3.0,18.257537,-6.257537,DR_0103
3,0104,VENADERO,1.9,0.9,1.733406,-0.733406,DR_0104
4,0105,VALLE DE CALVILLO,25.0,2.0,40.710928,-17.710928,DR_0105


In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
df = df.rename(columns={
    "CLAVE": "clave", "ACUÍFERO": "acuifero", "R": "recarga_hm3",
    "DNC": "dnc_hm3", "VEAS": "extraccion_hm3", "DMA": "dma_hm3",
    "DOCUMENTO": "documento",
})
df["acuifero"] = df["acuifero"].str.title()

# Validar: DMA debe ser igual a R - DNC - VEAS
df["dma_calculado"] = df["recarga_hm3"] - df["dnc_hm3"] - df["extraccion_hm3"]
print("Diferencia máxima:", (df["dma_hm3"] - df["dma_calculado"]).abs().max())

# Columnas nuevas
df["ratio_extraccion_recarga"] = (df["extraccion_hm3"] / df["recarga_hm3"]).round(2)
df["sobreexplotado"] = df["dma_hm3"] < 0

df = df.drop(columns="dma_calculado")
df.to_csv("../data/clean/acuiferos_ags_limpio.csv", index=False, encoding="utf-8-sig")
df

Diferencia máxima: 3.552713678800501e-15


,clave,acuifero,recarga_hm3,dnc_hm3,extraccion_hm3,dma_hm3,documento,ratio_extraccion_recarga,sobreexplotado
0,0101,Valle De Aguascalientes,249.6,2.4,342.958309,-95.758309,DR_0101,1.37,True
1,0102,Valle De Chicalote,35.0,0.0,48.796081,-13.796081,DR_0102,1.39,True
2,0103,El Llano,15.0,3.0,18.257537,-6.257537,DR_0103,1.22,True
3,0104,Venadero,1.9,0.9,1.733406,-0.733406,DR_0104,0.91,True
4,0105,Valle De Calvillo,25.0,2.0,40.710928,-17.710928,DR_0105,1.63,True
